# Multi-Agent Workflow Tutorial

This tutorial shows you how to build a **multi-agent system** using Llama 4 models. You'll learn how to orchestrate multiple specialized agents that collaborate to complete complex tasks.

## What You'll Learn

- Design and implement specialized agents with distinct roles
- Define tool schemas for each agent
- Build an orchestration layer for agent communication
- Execute a complete multi-agent workflow

## Use Case: Research Assistant System

We'll build a **Research Assistant** with three collaborating agents:

1. **Researcher Agent**: Searches for and summarizes information
2. **Critic Agent**: Evaluates claims and identifies gaps
3. **Writer Agent**: Synthesizes findings into a coherent report

This is part (3/3) in the tool calling series, building on concepts from Tool_Calling_101 and Tool_Calling_201.

## Setup

We'll use the Llama API for this tutorial. You can also use Groq, Together AI, or a local deployment with Ollama.

In [ ]:
# Install dependencies
# !pip install openai requests

In [ ]:
import os
import json
from typing import List, Dict, Any, Callable, Optional
from dataclasses import dataclass, field
from enum import Enum

# Set your API key
# Option 1: Llama API
os.environ["LLAMA_API_KEY"] = ""  # Get from https://llama.developer.meta.com

# Option 2: Groq (uncomment to use)
# os.environ["GROQ_API_KEY"] = ""

In [ ]:
from openai import OpenAI

# Configure client for Llama API (OpenAI-compatible)
client = OpenAI(
    api_key=os.environ.get("LLAMA_API_KEY"),
    base_url="https://api.llama.meta.com/v1"  # Llama API endpoint
)

# Alternative: Use Groq for faster inference
# from groq import Groq
# client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

MODEL = "llama-4-scout-17b-16e-instruct"  # or "llama-3.3-70b-versatile" for Groq

## Part 1: Define the Agent Architecture

Each agent has:
- A **role** (researcher, critic, writer)
- A **system prompt** defining its behavior
- A set of **tools** it can use
- An **execute** method to process messages

In [ ]:
class AgentRole(Enum):
    RESEARCHER = "researcher"
    CRITIC = "critic"
    WRITER = "writer"
    ORCHESTRATOR = "orchestrator"


@dataclass
class Message:
    """A message in the agent conversation."""
    role: str  # 'user', 'assistant', 'system', 'tool'
    content: str
    name: Optional[str] = None  # Agent name for multi-agent context
    tool_call_id: Optional[str] = None


@dataclass
class Tool:
    """A tool that an agent can use."""
    name: str
    description: str
    parameters: Dict[str, Any]
    function: Callable

    def to_schema(self) -> Dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters
            }
        }

In [ ]:
@dataclass
class Agent:
    """Base agent class for multi-agent workflows."""
    name: str
    role: AgentRole
    system_prompt: str
    tools: List[Tool] = field(default_factory=list)
    memory: List[Message] = field(default_factory=list)

    def execute(self, user_message: str, context: Optional[Dict] = None) -> str:
        """Execute the agent with a user message and optional context."""
        messages = [
            {"role": "system", "content": self._build_system_prompt(context)}
        ]
        
        # Add memory/history
        for msg in self.memory:
            messages.append({"role": msg.role, "content": msg.content})
        
        # Add current message
        messages.append({"role": "user", "content": user_message})
        
        # Build tool schemas if agent has tools
        tools = [tool.to_schema() for tool in self.tools] if self.tools else None
        
        # Call the model
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto" if tools else None,
            max_tokens=2048,
            temperature=0.7
        )
        
        assistant_message = response.choices[0].message
        
        # Handle tool calls if present
        if assistant_message.tool_calls:
            return self._handle_tool_calls(assistant_message, messages)
        
        # Store in memory
        result = assistant_message.content
        self.memory.append(Message(role="user", content=user_message))
        self.memory.append(Message(role="assistant", content=result))
        
        return result

    def _build_system_prompt(self, context: Optional[Dict] = None) -> str:
        """Build the system prompt with optional context."""
        prompt = self.system_prompt
        if context:
            context_str = "\n\nContext from other agents:\n"
            for agent_name, agent_output in context.items():
                context_str += f"\n[{agent_name}]:\n{agent_output}\n"
            prompt += context_str
        return prompt

    def _handle_tool_calls(self, assistant_message, messages: List[Dict]) -> str:
        """Execute tool calls and return final response."""
        messages.append(assistant_message.model_dump())
        
        for tool_call in assistant_message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            
            # Find and execute the tool
            tool = next((t for t in self.tools if t.name == tool_name), None)
            if tool:
                result = tool.function(**tool_args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                })
        
        # Get final response after tool execution
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            max_tokens=2048,
            temperature=0.7
        )
        
        return final_response.choices[0].message.content

    def clear_memory(self):
        """Clear agent memory."""
        self.memory = []

## Part 2: Define Tools for Each Agent

Each agent has specialized tools for its role.

In [ ]:
# Simulated tools (in production, connect to real APIs)

def search_papers(query: str, max_results: int = 5) -> str:
    """Simulated paper search - replace with arXiv API in production."""
    # Mock results for demonstration
    papers = [
        {"title": f"Research on {query} - Study {i+1}", 
         "abstract": f"This paper investigates {query} and finds significant results...",
         "year": 2024 - i}
        for i in range(min(max_results, 3))
    ]
    return json.dumps(papers, indent=2)


def search_web(query: str) -> str:
    """Simulated web search - replace with Brave Search API in production."""
    return f"Web search results for '{query}': Found relevant information about the topic including recent developments and key insights."


def analyze_claims(claims: str) -> str:
    """Analyze claims for validity - simulated."""
    return f"Analysis of claims: The provided claims appear to be partially supported by evidence. Key considerations: 1) Methodology appears sound, 2) Sample size should be verified, 3) Conclusions are reasonable but may need caveats."


def check_sources(sources: str) -> str:
    """Check source reliability - simulated."""
    return f"Source reliability check: The sources provided appear to be from reputable publications. Recommendation: Cross-reference with additional primary sources."


def format_report(content: str, style: str = "technical") -> str:
    """Format content into a structured report."""
    return f"## Formatted Report ({style} style)\n\n{content}\n\n---\n*Report generated by Writer Agent*"

In [ ]:
# Define tools for Researcher Agent
researcher_tools = [
    Tool(
        name="search_papers",
        description="Search academic papers on a given topic. Returns paper titles, abstracts, and years.",
        parameters={
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query for finding papers"},
                "max_results": {"type": "integer", "description": "Maximum number of results (default: 5)"}
            },
            "required": ["query"]
        },
        function=search_papers
    ),
    Tool(
        name="search_web",
        description="Search the web for recent information on a topic.",
        parameters={
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The web search query"}
            },
            "required": ["query"]
        },
        function=search_web
    )
]

# Define tools for Critic Agent
critic_tools = [
    Tool(
        name="analyze_claims",
        description="Analyze the validity of research claims.",
        parameters={
            "type": "object",
            "properties": {
                "claims": {"type": "string", "description": "The claims to analyze"}
            },
            "required": ["claims"]
        },
        function=analyze_claims
    ),
    Tool(
        name="check_sources",
        description="Check the reliability of cited sources.",
        parameters={
            "type": "object",
            "properties": {
                "sources": {"type": "string", "description": "The sources to check"}
            },
            "required": ["sources"]
        },
        function=check_sources
    )
]

# Define tools for Writer Agent
writer_tools = [
    Tool(
        name="format_report",
        description="Format content into a structured report with a specific style.",
        parameters={
            "type": "object",
            "properties": {
                "content": {"type": "string", "description": "The content to format"},
                "style": {"type": "string", "enum": ["technical", "executive", "blog"], "description": "Report style"}
            },
            "required": ["content"]
        },
        function=format_report
    )
]

## Part 3: Create the Specialized Agents

In [ ]:
# Create Researcher Agent
researcher_agent = Agent(
    name="Researcher",
    role=AgentRole.RESEARCHER,
    system_prompt="""You are an expert Research Agent specialized in finding and summarizing information.

Your responsibilities:
1. Search for relevant academic papers and web resources
2. Extract key findings and summarize them clearly
3. Identify the most important and relevant information
4. Cite your sources properly

Always use your tools to gather information before responding. Be thorough but concise.""",
    tools=researcher_tools
)

# Create Critic Agent
critic_agent = Agent(
    name="Critic",
    role=AgentRole.CRITIC,
    system_prompt="""You are a Critical Analysis Agent specialized in evaluating research quality.

Your responsibilities:
1. Evaluate the validity of claims and arguments
2. Check source reliability and credibility
3. Identify gaps, biases, or weaknesses in research
4. Suggest areas that need more investigation

Be constructive but thorough in your criticism. Focus on improving the quality of the final output.""",
    tools=critic_tools
)

# Create Writer Agent
writer_agent = Agent(
    name="Writer",
    role=AgentRole.WRITER,
    system_prompt="""You are a Professional Writer Agent specialized in creating polished reports.

Your responsibilities:
1. Synthesize research findings into coherent narratives
2. Address criticism and incorporate improvements
3. Create well-structured, readable reports
4. Maintain appropriate tone for the target audience

Transform raw research and feedback into a polished final product.""",
    tools=writer_tools
)

print(" Created 3 specialized agents: Researcher, Critic, Writer")

## Part 4: Build the Orchestrator

The orchestrator coordinates the agents and manages the workflow.

In [ ]:
@dataclass
class WorkflowResult:
    """Result of a multi-agent workflow execution."""
    topic: str
    research_output: str
    critique_output: str
    final_report: str
    iterations: int


class MultiAgentOrchestrator:
    """Orchestrates multi-agent workflows."""
    
    def __init__(self, researcher: Agent, critic: Agent, writer: Agent):
        self.researcher = researcher
        self.critic = critic
        self.writer = writer
    
    def execute_research_workflow(self, topic: str, max_iterations: int = 2) -> WorkflowResult:
        """
        Execute the full research workflow:
        1. Researcher gathers information
        2. Critic evaluates findings
        3. Writer produces final report
        """
        print(f"\n{'='*60}")
        print(f" Starting Research Workflow: {topic}")
        print(f"{'='*60}\n")
        
        # Clear agent memories for fresh start
        self.researcher.clear_memory()
        self.critic.clear_memory()
        self.writer.clear_memory()
        
        # Step 1: Research
        print(" Step 1: Researcher Agent gathering information...")
        research_prompt = f"Research the following topic thoroughly: {topic}. Find relevant papers and web sources. Provide a comprehensive summary of the key findings."
        research_output = self.researcher.execute(research_prompt)
        print(f"\n[Researcher Output]:\n{research_output[:500]}...\n")
        
        # Step 2: Critique
        print(" Step 2: Critic Agent evaluating findings...")
        critique_prompt = f"Evaluate the following research findings for accuracy, completeness, and potential issues:\n\n{research_output}"
        critique_output = self.critic.execute(critique_prompt)
        print(f"\n[Critic Output]:\n{critique_output[:500]}...\n")
        
        # Step 3: Refine (optional iteration)
        current_research = research_output
        iterations = 1
        
        if "need more" in critique_output.lower() or "missing" in critique_output.lower():
            print(" Critic identified gaps - Researcher addressing feedback...")
            refined_prompt = f"Based on this feedback, improve your research:\n\nFeedback: {critique_output}\n\nOriginal research: {research_output}"
            current_research = self.researcher.execute(refined_prompt)
            iterations += 1
        
        # Step 4: Write final report
        print("️ Step 3: Writer Agent creating final report...")
        write_prompt = f"""Create a polished research report based on:

Research Findings:
{current_research}

Critical Feedback Addressed:
{critique_output}

Create a comprehensive, well-structured report on: {topic}"""
        
        context = {
            "Researcher": current_research,
            "Critic": critique_output
        }
        final_report = self.writer.execute(write_prompt, context=context)
        
        print(f"\n{'='*60}")
        print(" Workflow Complete!")
        print(f"{'='*60}\n")
        
        return WorkflowResult(
            topic=topic,
            research_output=current_research,
            critique_output=critique_output,
            final_report=final_report,
            iterations=iterations
        )

## Part 5: Execute the Multi-Agent Workflow

Let's run our research assistant system on a real topic!

In [ ]:
# Create the orchestrator
orchestrator = MultiAgentOrchestrator(
    researcher=researcher_agent,
    critic=critic_agent,
    writer=writer_agent
)

# Execute workflow on a topic
topic = "Latest advances in Llama model fine-tuning techniques"
result = orchestrator.execute_research_workflow(topic)

In [ ]:
# Display the final report
print("\n" + "="*60)
print(" FINAL RESEARCH REPORT")
print("="*60 + "\n")
print(result.final_report)

In [ ]:
# Workflow statistics
print(f"\n Workflow Statistics:")
print(f"   Topic: {result.topic}")
print(f"   Iterations: {result.iterations}")
print(f"   Research length: {len(result.research_output)} chars")
print(f"   Critique length: {len(result.critique_output)} chars")
print(f"   Final report length: {len(result.final_report)} chars")

## Part 6: Advanced Patterns

### 6.1 Parallel Agent Execution

When agents don't depend on each other, run them in parallel for faster results.

In [ ]:
import concurrent.futures

def parallel_research(topics: List[str], researcher: Agent) -> Dict[str, str]:
    """Execute research on multiple topics in parallel."""
    results = {}
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        future_to_topic = {
            executor.submit(researcher.execute, f"Research: {topic}"): topic 
            for topic in topics
        }
        
        for future in concurrent.futures.as_completed(future_to_topic):
            topic = future_to_topic[future]
            try:
                results[topic] = future.result()
            except Exception as e:
                results[topic] = f"Error: {str(e)}"
    
    return results

# Example: Research multiple topics in parallel
# topics = ["LoRA fine-tuning", "Quantization methods", "RLHF training"]
# parallel_results = parallel_research(topics, researcher_agent)

### 6.2 Agent Communication Patterns

Different patterns for agent collaboration:

In [ ]:
def debate_pattern(agent_a: Agent, agent_b: Agent, topic: str, rounds: int = 3) -> List[Dict]:
    """
    Two agents debate a topic, refining their positions through multiple rounds.
    Good for: Exploring different perspectives, finding edge cases.
    """
    debate_history = []
    current_argument = f"Take a position on: {topic}"
    
    for round in range(rounds):
        # Agent A responds
        response_a = agent_a.execute(current_argument)
        debate_history.append({"agent": agent_a.name, "round": round + 1, "response": response_a})
        
        # Agent B responds to A's argument
        counter_prompt = f"Counter or build upon this argument: {response_a}"
        response_b = agent_b.execute(counter_prompt)
        debate_history.append({"agent": agent_b.name, "round": round + 1, "response": response_b})
        
        current_argument = f"Respond to: {response_b}"
    
    return debate_history


def chain_pattern(agents: List[Agent], initial_input: str) -> List[Dict]:
    """
    Pass output from one agent to the next in a chain.
    Good for: Processing pipelines, sequential refinement.
    """
    chain_outputs = []
    current_input = initial_input
    
    for agent in agents:
        output = agent.execute(current_input)
        chain_outputs.append({"agent": agent.name, "input": current_input, "output": output})
        current_input = output
    
    return chain_outputs


print(" Advanced patterns defined: debate_pattern, chain_pattern")

## Summary

In this tutorial, you learned how to:

1. **Design a multi-agent architecture** with specialized roles (Researcher, Critic, Writer)
2. **Define tools** for each agent using JSON schemas compatible with Llama's function calling
3. **Build an orchestrator** to coordinate agent interactions
4. **Execute workflows** that combine multiple agents for complex tasks
5. **Apply advanced patterns** like parallel execution and debate/chain patterns

### Next Steps

- Connect to real APIs (arXiv, Brave Search, etc.) for production use
- Add persistence for agent memory across sessions
- Implement error handling and retry logic
- Explore more complex workflows with conditional branching

### Resources

- [Llama API Documentation](https://llama.developer.meta.com/docs)
- [Tool Calling 101](./Tool_Calling_101.ipynb) - Basics of tool calling
- [Tool Calling 201](./Tool_Calling_201.ipynb) - Advanced tool patterns